# Hostile Testing Phase 9 - Core Logging & Git Analysis

## Purpose
Test remaining core logging and git analysis functions with hostile inputs

## Goal
Add 8+ functions, reach 8%+ coverage

In [ ]:
import pandas as pd
import tempfile
from pathlib import Path

test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Phase 9 test harness loaded - CORE LOGGING & GIT")

## Section 1: Core Logging - Additional Log Functions

In [ ]:
from siege_utilities import (
    log_error, log_debug, log_critical, log_exception
)

# Test log_error with hostile messages
hostile_messages = [
    None,
    "",
    "'; DROP TABLE logs; --",
    "<script>alert(1)</script>",
    "Error: $(cat /etc/passwd)",
    "Error: `whoami`",
    "../../../etc/passwd",
    "A" * 10000,
    "msg\x00null",
    "数据库错误",  # Unicode
]

for msg in hostile_messages:
    success, result, error = hostile_test(log_error, "hostile_msg", msg)
    record_test(
        "log_error",
        "core.logging",
        "hostile_messages",
        success,
        error,
        "critical" if "$(" in str(msg) or "`" in str(msg) or "DROP TABLE" in str(msg) else "high" if "../" in str(msg) else "medium"
    )

# Test log_debug
for msg in hostile_messages[:5]:
    success, result, error = hostile_test(log_debug, "hostile_msg", msg)
    record_test("log_debug", "core.logging", "hostile_messages", success, error, "medium")

# Test log_critical
for msg in hostile_messages[:5]:
    success, result, error = hostile_test(log_critical, "hostile_msg", msg)
    record_test("log_critical", "core.logging", "hostile_messages", success, error, "high")

print(f"Completed {len([r for r in test_results['module'] if 'logging' in r])} logging tests")

## Section 2: Git Analysis - Branch Operations

In [ ]:
from siege_utilities import (
    analyze_branch_status, get_current_branch, get_remote_branches
)

# Test analyze_branch_status with hostile repo paths
hostile_repo_paths = [
    None,
    "",
    "'; DROP TABLE repos; --",
    "<script>alert(1)</script>",
    "../../../etc/passwd",
    "/etc/shadow",
    "repo; rm -rf /",
    "$(cat /etc/passwd)",
    "`whoami`",
    "A" * 10000,
    "path\x00null",
    ".git",  # Valid relative
]

for repo_path in hostile_repo_paths:
    success, result, error = hostile_test(analyze_branch_status, "hostile_repo", repo_path)
    record_test(
        "analyze_branch_status",
        "git.branch_analyzer",
        "hostile_repo_paths",
        success,
        error,
        "critical" if "DROP TABLE" in str(repo_path) or "rm -rf" in str(repo_path) or "$(" in str(repo_path) or "`" in str(repo_path) else "high" if "../" in str(repo_path) or "/etc/" in str(repo_path) else "medium"
    )

# Test get_current_branch with hostile paths
for repo_path in hostile_repo_paths[:8]:
    success, result, error = hostile_test(get_current_branch, "hostile_repo", repo_path)
    record_test(
        "get_current_branch",
        "git.branch_analyzer",
        "hostile_repo_paths",
        success,
        error,
        "high" if "../" in str(repo_path) or "DROP" in str(repo_path) else "medium"
    )

print(f"Completed {len([r for r in test_results['module'] if 'git' in r])} git tests")

## Section 3: Admin - Client Management

In [ ]:
from siege_utilities import (
    list_client_profiles, get_client_profile, save_client_profile
)

test_dir = Path(tempfile.mkdtemp())
try:
    # Test list_client_profiles with hostile directory paths
    hostile_dirs = [
        None,
        "",
        "../../../etc",
        "/etc/shadow",
        "'; DROP TABLE dirs; --",
        "dir; rm -rf /",
        "A" * 10000,
        str(test_dir),  # Valid
    ]
    
    for dir_path in hostile_dirs:
        success, result, error = hostile_test(list_client_profiles, "hostile_dir", dir_path)
        record_test(
            "list_client_profiles",
            "admin.profile_manager",
            "hostile_dirs",
            success,
            error,
            "high" if "../" in str(dir_path) or "/etc/" in str(dir_path) or "DROP" in str(dir_path) else "medium"
        )

finally:
    import shutil
    shutil.rmtree(test_dir, ignore_errors=True)

print(f"Completed {len([r for r in test_results['module'] if 'admin' in r])} admin tests")

## Section 4: Generate Phase 9 Results

In [ ]:
df = pd.DataFrame(test_results)

print("\n" + "="*80)
print("PHASE 9 HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nTests run: {len(df)}")
print(f"Passed: {df['passed'].sum()}")
print(f"Failed: {(~df['passed']).sum()}")
print(f"Pass rate: {df['passed'].sum() / len(df) * 100:.1f}%")

unique = df['function'].nunique()
print(f"\nUnique functions: {unique}")
print(f"Phase 9 coverage: {unique}/751 = {unique/751*100:.1f}%")

print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
summary = df.groupby('module').agg({'passed': ['sum', 'count']})
summary.columns = ['passed', 'total']
summary['pass_rate'] = (summary['passed'] / summary['total'] * 100).round(1)
print(summary)

failures = df[~df['passed']]
if len(failures) > 0:
    print("\n" + "="*80)
    print("FAILURES BY SEVERITY")
    print("="*80)
    print(failures.groupby('severity').size())
    
    critical_high = failures[failures['severity'].isin(['critical', 'high'])]
    if len(critical_high) > 0:
        print("\n" + "="*80)
        print("CRITICAL/HIGH FAILURES (First 5)")
        print("="*80)
        for idx, row in critical_high.head(5).iterrows():
            print(f"\n{row['function']} ({row['severity']})")
            print(f"  Module: {row['module']}")
            print(f"  Error: {row['error_message'][:100]}")
else:
    print("\n✅ NO FAILURES")

df.to_csv("hostile_testing_phase9_results.csv", index=False)
print(f"\nSaved: hostile_testing_phase9_results.csv")

print("\n" + "="*80)
print("FUNCTIONS TESTED")
print("="*80)
for func in df['function'].unique():
    tests = df[df['function'] == func]
    passed = tests['passed'].sum()
    total = len(tests)
    print(f"{func}: {passed}/{total} ({passed/total*100:.0f}%)")

print("\n" + "="*80)
print("CUMULATIVE COVERAGE (Phases 1-9)")
print("="*80)
total_unique = 51 + unique
print(f"Total unique functions: ~{total_unique}")
print(f"Cumulative coverage: ~{total_unique}/751 = {total_unique/751*100:.1f}%")
print(f"\n🎯 Progress to 25% goal: {total_unique}/188 = {total_unique/188*100:.1f}%")